# Readout mitigation, gate twirling and ZNE on ibm_phoenix (same circuit as PEC)

Prof. Goan (13 Oct): run readout mitigation, gate twirling, and ZNE on ibm_phoenix as well.

**Circuit.** The ISA circuit of the PEC runs (10 Oct batch `REF_BATCH`, seed `REF_SEED`, 84 CZ) is loaded from QPY,
not re-transpiled, so these rows and the 15 PEC runs measure exactly the same circuit (OTC group 0, rep 0; target +0.8;
observables Z0, Z4, Z6, X5, X7).

**Methods.** The options are copied from `Comparsion.ipynb` (ibm_marrakesh, 4 May), i.e. the rows of Table 10:

| key | Table 10 label | options |
|---|---|---|
| `baseline` | Readout mitigation only | `EstimatorOptions()` (resilience level 1 = TREX) |
| `twirling` | Twirling (300 inst.) | level 1 + `twirling.enable_gates = True`, `num_randomizations = 300` |
| `zne` | ZNE (32 twirling inst.) | `resilience_level = 2` (ZNE + gate twirling, default 32 randomizations) |
| `zne_twirling` | ZNE (300 twirling inst.) | level 2 + `twirling.enable_gates = True`, `num_randomizations = 300` |

Shots are left at the defaults, as on ibm_marrakesh (`default_precision` 1/64, i.e. about 4,000 shots).
Only `max_execution_time` is added (a safety cap; it does not change the mitigation).

**Repetitions.** `N_REPEATS` independent runs per method, all submitted together in one batch.
Note: on ibm_marrakesh the three repetitions were three OTC parameter sets (rep 0, 1, 2) transpiled separately;
here the three runs repeat the PEC circuit (rep 0), as for the 15 PEC runs.

**Files.** Put this notebook in the PEC folder: it needs `qru_otc_multi_obs_20260503_205659.csv` and the three
10 Oct files (`pec_isa_circuit_…qpy`, `pec_observables_…json`, `pec_repeats_circuits_…json`).
Results go to `mitigation_results_ibm_phoenix.csv` (no job ids); job ids go to the pending JSON and `pec_job_ids.txt`.

Run order: Cells 1–5 (no QPU) → Cell 6 SUBMIT once → Cell 7 COLLECT (re-run until all DONE) → Cell 8 analysis.

In [ ]:
# ── Cell 1: settings ──
from datetime import datetime, timezone
DRY_RUN       = False
SERVICE_NAME  = "NTU"
INSTANCE      = "<YOUR_INSTANCE_NAME>"
BACKEND_NAME  = "ibm_phoenix"
REF_BATCH     = "20261010_021000"   # 10 Oct batch whose ISA circuit is reused (same as the PEC runs)
REF_SEED      = 0
CZ_ERR_MAX    = 0.01                # only used to rebuild the 10 Oct file names
GRP_ID, REP   = 0, 0
METHODS       = ['baseline', 'twirling', 'zne', 'zne_twirling']
N_REPEATS     = 3                   # independent runs per method
EXEC_MODE     = "batch"             # "batch" or "session"
MAX_EXEC_S    = 600                 # cap per job (safety only)

In [ ]:
# ── Cell 2: imports + circuit (from Comparsion.ipynb) ──
"""
OTC-based Observable Comparison Experiment
比較 SV, Noise Model, FakeMarrakesh, ibm_marrakesh (Baseline/ZNE/Twirling/ZNE+Twirling/QESEM)
Group 1 (target=+0.8): Z0, Z4, Z6, X5, X7
Group 2 (target=-0.8): Z1, Z5, Z7, X4, X6
"""

import numpy as np
import pandas as pd
import csv, time
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
from qiskit_ibm_runtime import QiskitRuntimeService, Session
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

import os, json, glob
from qiskit import qpy
from qiskit.quantum_info import PauliList
from qiskit_ibm_runtime import Batch, Session
from qiskit.transpiler import Target

def build_qru_circuit(data_input, hidden_input, params):
    """
    data_input:   numpy array, shape (8,)
    hidden_input: numpy array, shape (8,), 初始為全零
    params:       numpy array, shape (131,)
    """

    data   = QuantumRegister(4, 'data')
    hidden = QuantumRegister(4, 'hidden')
    anc    = QuantumRegister(2, 'anc')
    qc = QuantumCircuit(data, hidden, anc)

    # ==================
    # 1. ENCODING
    # ==================
    # Data: params[0:16], 每 qubit 4 個參數
    qc.barrier(label='ENC')
    for i in range(4):
        rx_angle = data_input[2*i] * params[4*i]     + params[4*i + 1]
        ry_angle = data_input[2*i + 1] * params[4*i + 2] + params[4*i + 3]
        qc.rx(rx_angle, data[i])
        qc.ry(ry_angle, data[i])

    # Hidden: params[16:32]
    for i in range(4):
        rx_angle = hidden_input[2*i] * params[16 + 4*i]     + params[16 + 4*i + 1]
        ry_angle = hidden_input[2*i + 1] * params[16 + 4*i + 2] + params[16 + 4*i + 3]
        qc.rx(rx_angle, hidden[i])
        qc.ry(ry_angle, hidden[i])

    # 循環 CX on data: 0→1→2→3→0
    for i in range(4):
        qc.cx(data[i], data[(i+1) % 4])

    # 循環 CX on hidden: 4→5→6→7→4
    for i in range(4):
        qc.cx(hidden[i], hidden[(i+1) % 4])

    # ==================
    # 2. UPDATE & RESET
    # ==================
    qc.barrier(label='UPD')
    qc.cx(data[0], hidden[0])
    qc.cswap(hidden[0], hidden[1], anc[0])

    # Rot on hidden[0]: params[32:35]
    qc.rz(params[32], hidden[0])
    qc.ry(params[33], hidden[0])
    qc.rz(params[34], hidden[0])

    qc.cswap(hidden[0], hidden[3], anc[1])
    # ancilla 保持 |0⟩，不需 reset

    # ==================
    # 3. VARIATIONAL LAYERS（4 層）
    # ==================
    idx = 35
    for layer in range(4):
        qc.barrier(label=f'VAR{layer+1}')
        # Rot on all non-ancilla qubits
        for reg in [data, hidden]:
            for i in range(4):
                qc.rz(params[idx],     reg[i]); idx += 1
                qc.ry(params[idx],     reg[i]); idx += 1
                qc.rz(params[idx],     reg[i]); idx += 1

        # 循環 CX on all non-ancilla: data[0..3] → hidden[0..3] → back
        all_q = [data[i] for i in range(4)] + [hidden[i] for i in range(4)]
        for i in range(8):
            qc.cx(all_q[i], all_q[(i+1) % 8])

    return qc

In [ ]:
# ── Cell 3: OTC parameters and observables (from Comparsion.ipynb) ──
# ── 設定 ──────────────────────────────────────────────────────
CALIB_CSV  = '../error_mitigation_comparison/qru_otc_multi_obs_20260503_205659.csv'
OUT_CSV = None
N_REPS     = 3      # 每組電路重複 3 次
N_SHOTS    = 4000

# ── Observable 分組 ───────────────────────────────────────────
# (qubit, pauli, obs_name, out_idx, group_id, target_val)
OBS_GROUPS = {
    0: {  # Group 1: target = +0.8
        'target': +0.8,
        'obs': [
            (0, 'Z', 'Z0',  0),
            (4, 'Z', 'Z4',  8),
            (6, 'Z', 'Z6', 12),
            (5, 'X', 'X5', 11),
            (7, 'X', 'X7', 15),
        ]
    },
    1: {  # Group 2: target = -0.8
        'target': -0.8,
        'obs': [
            (1, 'Z', 'Z1',  2),
            (5, 'Z', 'Z5', 10),
            (7, 'Z', 'Z7', 14),
            (4, 'X', 'X4',  9),
            (6, 'X', 'X6', 13),
        ]
    },
}

# ── OTC 資料庫讀取 ────────────────────────────────────────────
param_cols = [c for c in pd.read_csv(CALIB_CSV, nrows=0).columns if c.startswith('p')]
calib_df   = pd.read_csv(CALIB_CSV)

def get_params(group_id, target_val, rep_id):
    """從 OTC 資料庫取得電路參數"""
    obs_name = OBS_GROUPS[group_id]['obs'][0][2]  # 用第一個 obs 查詢
    mask = (
        (calib_df['group_id'] == group_id) &
        (calib_df['target_val'] == target_val) &
        (calib_df['rep_id'] == rep_id) &
        (calib_df['obs_name'] == obs_name)
    )
    rows = calib_df[mask]
    if len(rows) == 0:
        return None, None
    row     = rows.iloc[0]
    params  = row[param_cols].values.astype(float)
    sv_vals = {}
    for _, r in calib_df[(calib_df['group_id']==group_id) &
                          (calib_df['target_val']==target_val) &
                          (calib_df['rep_id']==rep_id)].iterrows():
        sv_vals[r['obs_name']] = r['sv_val']
    return params, sv_vals

# ── Observable 建構 ───────────────────────────────────────────
def make_obs_list(group_id, n=10):
    obs_list = []
    for q, p, name, _ in OBS_GROUPS[group_id]['obs']:
        s = ['I'] * n
        s[n-1-q] = p
        obs_list.append(SparsePauliOp(''.join(s)))
    return obs_list


RESULTS_CSV = f'pec_fixedshots_results_{BACKEND_NAME}.csv'
def status_name(job):
    s = job.status()
    return s if isinstance(s, str) else s.name
def cz_error_sum(target, circ):
    return float(sum(target[ins.operation.name][tuple(circ.find_bit(q).index for q in ins.qubits)].error
                     for ins in circ.data if ins.operation.num_qubits == 2 and ins.operation.name != 'barrier'))

RESULTS_CSV = f'mitigation_results_{BACKEND_NAME}.csv'
# Table 10 (ibm_marrakesh, 4 May) and PEC (ibm_phoenix, 15 runs): mean deviation from SV in %
TABLE10_MARRAKESH = {'baseline': -78.2, 'twirling': -83.4, 'zne': -70.9, 'zne_twirling': -65.2}
PEC_PHOENIX = -6.0
LABELS = {'baseline': 'Readout mitigation only', 'twirling': 'Twirling (300 inst.)',
          'zne': 'ZNE (32 twirling inst.)', 'zne_twirling': 'ZNE (300 twirling inst.)'}

In [ ]:
# ── Cell 4: backend, load the 10 Oct ISA circuit, verification (no QPU) ──
if DRY_RUN:
    backend = AerSimulator.from_backend(FakeMarrakesh()); calib = FakeMarrakesh().target
else:
    service = QiskitRuntimeService(name=SERVICE_NAME, instance=INSTANCE)
    backend = service.backend(BACKEND_NAME); calib = backend.target
target_val = OBS_GROUPS[GRP_ID]['target']
obs_names  = [o[2] for o in OBS_GROUPS[GRP_ID]['obs']]
params, sv_vals = get_params(GRP_ID, target_val, REP)
sv_arr = np.array([sv_vals[n] for n in obs_names])
logical_qc = build_qru_circuit(np.zeros(8), np.zeros(8), params)

def reduce_to_active(circ, ops):
    act = {circ.find_bit(q).index for ins in circ.data
           if ins.operation.name not in ('barrier', 'measure', 'delay', 'reset') for q in ins.qubits}
    for op in ops:
        act |= set(np.nonzero(np.any(op.paulis.z | op.paulis.x, axis=0))[0].tolist())
    act = sorted(act); m = {q: i for i, q in enumerate(act)}
    small = QuantumCircuit(len(act))
    for ins in circ.data:
        if ins.operation.name not in ('barrier', 'measure', 'delay', 'reset'):
            small.append(ins.operation, [m[circ.find_bit(q).index] for q in ins.qubits])
    return small, [SparsePauliOp(PauliList.from_symplectic(op.paulis.z[:, act], op.paulis.x[:, act]), op.coeffs)
                   for op in ops]

tag = f'{BACKEND_NAME}_seed{REF_SEED}_excl{int(CZ_ERR_MAX*100)}pct_{REF_BATCH}'
CIRCUIT_FILE  = f'pec_isa_circuit_{tag}.qpy'
OBS_FILE      = f'pec_observables_{tag}.json'
CIRCUITS_JSON = f'pec_repeats_circuits_{BACKEND_NAME}_{REF_BATCH}.json'

if DRY_RUN and not os.path.exists(CIRCUIT_FILE):          # dry run only: make local stand-in files
    pm = generate_preset_pass_manager(target=calib, optimization_level=1, seed_transpiler=REF_SEED)
    qc = pm.run(logical_qc); obs = [ob.apply_layout(qc.layout) for ob in make_obs_list(GRP_ID)]
    qpy.dump(qc, open(CIRCUIT_FILE, 'wb'))
    json.dump({n: op.to_list() for n, op in zip(obs_names, obs)}, open(OBS_FILE, 'w'), default=str)
    json.dump({'circuits': [{'seed': REF_SEED, 'n_cz': qc.count_ops().get('cz', 0), 'depth': qc.depth(),
                             'sum_cz_error': cz_error_sum(calib, qc)}]}, open(CIRCUITS_JSON, 'w'))

isa_qc  = qpy.load(open(CIRCUIT_FILE, 'rb'))[0]
obs_raw = json.load(open(OBS_FILE))
isa_obs = [SparsePauliOp.from_list([(p, complex(c)) for p, c in obs_raw[n]]) for n in obs_names]
ref_info = {c['seed']: c for c in json.load(open(CIRCUITS_JSON))['circuits']}[REF_SEED]

small, small_ops = reduce_to_active(isa_qc, isa_obs)
sv_check = np.max(np.abs(np.asarray(StatevectorEstimator().run([(small, small_ops)]).result()[0].data.evs) - sv_arr))
assert sv_check < 1e-6, f"loaded circuit does not reproduce the SV reference ({sv_check:.1e})"
n_cz = isa_qc.count_ops().get('cz', 0)
assert n_cz == ref_info['n_cz'] and isa_qc.depth() == ref_info['depth'], "circuit differs from the 10 Oct summary"
print(f"{backend.name} | {CIRCUIT_FILE} | SV check {sv_check:.1e} | depth {isa_qc.depth()}, CZ {n_cz}")
print(f"summed CZ error: {ref_info['sum_cz_error']:.3f} at 10 Oct transpile -> {cz_error_sum(calib, isa_qc):.3f} now")

In [ ]:
# ── Cell 5: options (as in Comparsion.ipynb) and plan (no QPU) ──
def make_opts(method):
    o = EstimatorOptions()
    if method in ('zne', 'zne_twirling'):
        o.resilience_level = 2
    if method in ('twirling', 'zne_twirling'):
        o.twirling.enable_gates = True
        o.twirling.num_randomizations = 300
    if not DRY_RUN:
        o.max_execution_time = MAX_EXEC_S
    return o

plan = [(k, m) for k in range(N_REPEATS) for m in METHODS]       # interleaved: drift spreads over all methods
for m in METHODS:
    o = make_opts(m)
    print(f"{m:13s} resilience_level={o.resilience_level}  enable_gates={o.twirling.enable_gates}  "
          f"num_randomizations={o.twirling.num_randomizations}")
print(f"{len(plan)} jobs on {BACKEND_NAME}: " + ", ".join(f"{m}/r{k}" for k, m in plan))
if os.path.exists(RESULTS_CSV):
    print(f"note: {RESULTS_CSV} already has {len(pd.read_csv(RESULTS_CSV))} rows; new runs are appended")

In [ ]:
# ── Cell 6: SUBMIT (returns immediately; run once) ──
label = time.strftime('%Y%m%d_%H%M%S')
jobs, jobs_local = [], {}
if DRY_RUN:
    for k, m in plan:
        j = Estimator(mode=backend, options=make_opts(m)).run([(isa_qc, isa_obs)]); jobs_local[j.job_id()] = j
        jobs.append({'method': m, 'repeat': k, 'job_id': j.job_id()})
else:
    Mode = Batch if EXEC_MODE == "batch" else Session
    with Mode(backend=backend) as mode:
        ests = {m: Estimator(mode=mode, options=make_opts(m)) for m in METHODS}
        for k, m in plan:
            jobs.append({'method': m, 'repeat': k, 'job_id': ests[m].run([(isa_qc, isa_obs)]).job_id()})
pending = {'label': label, 'backend': BACKEND_NAME, 'instance': INSTANCE, 'exec_mode': EXEC_MODE,
           'submitted': time.strftime('%Y-%m-%d %H:%M:%S'), 'circuit_file': CIRCUIT_FILE,
           'ref_batch': REF_BATCH, 'ref_seed': REF_SEED, 'jobs': jobs}
with open(f'mitigation_pending_{BACKEND_NAME}_{label}.json', 'w') as f:      # local only (has job ids)
    json.dump(pending, f, indent=2)
with open('pec_job_ids.txt', 'a') as f:
    for j in jobs:
        f.write(f"{pending['submitted']} {BACKEND_NAME} mitigation_{label} seed{REF_SEED} "
                f"{j['method']} rep{j['repeat']} {j['job_id']}\n")
print(f"submitted {len(jobs)} job(s) in one {EXEC_MODE}; saved mitigation_pending_{BACKEND_NAME}_{label}.json")

In [ ]:
# ── Cell 7: COLLECT — re-run until all jobs are DONE (after a kernel restart: run Cells 1-3 first) ──
def parse_ts(s):
    if s is None: return None
    return datetime.fromisoformat(str(s).replace('Z', '+00:00'))
obs_names = [o[2] for o in OBS_GROUPS[GRP_ID]['obs']]
_, sv_vals = get_params(GRP_ID, OBS_GROUPS[GRP_ID]['target'], REP)
sv_arr = np.array([sv_vals[n] for n in obs_names])
done = pd.read_csv(RESULTS_CSV, dtype={'batch': str}) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
done_keys = set(zip(done['batch'], done['method'], done['repeat'])) if len(done) else set()
if not DRY_RUN:
    svc = QiskitRuntimeService(name=SERVICE_NAME, instance=INSTANCE)
    bk = svc.backend(BACKEND_NAME)
new_rows, counts, circ_cache = [], {}, {}
for pf in sorted(glob.glob(f'mitigation_pending_{BACKEND_NAME}_*.json')):
    pend = json.load(open(pf))
    for jd in pend['jobs']:
        key = (str(pend['label']), jd['method'], jd['repeat'])
        if key in done_keys:
            counts['saved'] = counts.get('saved', 0) + 1; continue
        if DRY_RUN and jd['job_id'] not in globals().get('jobs_local', {}):
            continue                                 # dry-run job from an earlier kernel
        job = jobs_local[jd['job_id']] if DRY_RUN else svc.job(jd['job_id'])
        if DRY_RUN: job.result()
        st = status_name(job); counts[st] = counts.get(st, 0) + 1
        if st != 'DONE':
            continue
        if pend['circuit_file'] not in circ_cache:
            circ_cache[pend['circuit_file']] = qpy.load(open(pend['circuit_file'], 'rb'))[0]
        circ = circ_cache[pend['circuit_file']]
        res = job.result(); meta = res[0].metadata; jmeta = res.metadata or {}
        evs, stds = np.asarray(res[0].data.evs), np.asarray(res[0].data.stds)
        rel = (evs - sv_arr) / sv_arr
        t_run, cz_run, qpu = None, np.nan, np.nan
        if not DRY_RUN:
            qpu = job.usage()
            t_run = parse_ts((job.metrics().get('timestamps') or {}).get('running'))
            if t_run is not None:
                cz_run = cz_error_sum(bk.target_history(datetime=t_run), circ)
        jtw, jres = jmeta.get('twirling', {}) or {}, jmeta.get('resilience', {}) or {}
        row = {'batch': str(pend['label']), 'method': jd['method'], 'repeat': jd['repeat'],
               'started_utc': t_run.isoformat() if t_run else None, 'sum_cz_error_at_start': cz_run,
               'shots': meta.get('shots'), 'num_randomizations': meta.get('num_randomizations'),
               'twirl_gates': jtw.get('enable_gates'), 'twirl_num_rand': jtw.get('num_randomizations'),
               'twirl_shots_per_rand': jtw.get('shots_per_randomization'),
               'zne': jres.get('zne_mitigation'), 'zne_noise_factors': str((jres.get('zne') or {}).get('noise_factors')),
               'measure_mitigation': jres.get('measure_mitigation'), 'qpu_s': qpu,
               'mean_dev_pct': 100 * rel.mean(), 'min_dev_pct': 100 * rel.min(), 'max_dev_pct': 100 * rel.max(),
               'mean_std': float(stds.mean())}
        row.update({f'ev_{n}': float(e) for n, e in zip(obs_names, evs)})
        row.update({f'std_{n}': float(s) for n, s in zip(obs_names, stds)})
        new_rows.append(row)
if new_rows:
    done = pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True)
    done.to_csv(RESULTS_CSV, index=False)        # no job ids in this file
print("status:", counts, f"| {len(new_rows)} new results saved to {RESULTS_CSV} ({len(done)} total)")
if len(done):
    print(done[['batch', 'method', 'repeat', 'shots', 'num_randomizations', 'twirl_gates', 'twirl_num_rand',
                'twirl_shots_per_rand', 'zne', 'zne_noise_factors', 'measure_mitigation', 'mean_dev_pct',
                'mean_std', 'qpu_s']].sort_values(['method', 'repeat'])
          .to_string(index=False, float_format=lambda v: f'{v:.3g}'))

In [ ]:
# ── Cell 8: analysis — ibm_phoenix vs Table 10 (ibm_marrakesh) and PEC (ibm_phoenix) ──
assert os.path.exists(RESULTS_CSV), "no results yet: run Cell 7 until jobs are DONE"
df = pd.read_csv(RESULTS_CSV, dtype={'batch': str})
ev_cols = [f'ev_{n}' for n in obs_names]
rows = []
for m in METHODS:
    d = df[df.method == m]
    if not len(d): continue
    per_obs = d[ev_cols].mean().values                    # mean over runs, per observable
    dev = 100 * ((per_obs - sv_arr) / sv_arr).mean()      # Table 10 convention: mean of signed relative deviations
    rows.append({'method': LABELS[m], 'runs': len(d), 'phoenix mean dev (%)': round(dev, 1),
                 'per-run range (%)': f"{d.mean_dev_pct.min():.1f} to {d.mean_dev_pct.max():.1f}",
                 'marrakesh, Table 10 (%)': TABLE10_MARRAKESH[m],
                 'shots': f"{int(d.shots.min()):,}–{int(d.shots.max()):,}" if d.shots.notna().all() else '-',
                 'QPU s/run': f"{d.qpu_s.mean():.0f}" if d.qpu_s.notna().all() else '-'})
rows.append({'method': 'PEC (15 runs)', 'runs': 15, 'phoenix mean dev (%)': PEC_PHOENIX,
             'per-run range (%)': '-9.7 to -3.2', 'marrakesh, Table 10 (%)': '-', 'shots': '0.56–2.31 M', 'QPU s/run': '77–121'})
print(pd.DataFrame(rows).to_string(index=False))

print("\nper observable on ibm_phoenix (mean ± run-to-run SD; SV reference in the first row):")
tab = pd.DataFrame({'SV': sv_arr}, index=obs_names).T
for m in METHODS:
    d = df[df.method == m]
    if len(d):
        tab.loc[LABELS[m]] = [f"{d[c].mean():.3f} ± {d[c].std(ddof=1):.3f}" if len(d) > 1 else f"{d[c].mean():.3f}"
                              for c in ev_cols]
print(tab.to_string())